In [1]:
#Fine-Tuning BERT

In [2]:
import pandas as pd
import numpy as np
import ast
import tensorflow as tf

In [3]:
mh=pd.read_csv("final.csv")

In [4]:
mh.sample(5)

,Unnamed: 0,input_ids,token_type_ids,attention_mask,status,status_encodes
1723,36838,"[101, 8816, 1996, 25218, 10439, 1045, 8046, 20...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",Normal,3.0
889,49074,"[101, 2054, 2003, 1996, 11234, 2090, 5554, 602...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",Stress,5.0
2245,40804,"[101, 14992, 2243, 28394, 2078, 2009, 2393, 20...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",Depression,2.0
1304,38831,"[101, 4931, 3124, 1045, 1049, 1037, 2310, 3089...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",Depression,2.0
2431,16166,"[101, 2021, 2009, 3280, 3892, 2061, 2182, 2003...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",Suicidal,6.0


In [5]:
mh.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2485 entries, 0 to 2484
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      2485 non-null   int64  
 1   input_ids       2469 non-null   object 
 2   token_type_ids  2469 non-null   object 
 3   attention_mask  2469 non-null   object 
 4   status          2471 non-null   object 
 5   status_encodes  2471 non-null   float64
dtypes: float64(1), int64(1), object(4)
memory usage: 116.6+ KB


In [6]:
mh=mh.dropna()

In [7]:
mh['input_ids']=mh['input_ids'].apply(ast.literal_eval)
mh['token_type_ids']=mh['token_type_ids'].apply(ast.literal_eval)
mh['attention_mask']=mh['attention_mask'].apply(ast.literal_eval)

In [8]:
iids=mh['input_ids'].tolist()

In [9]:
iidten=tf.convert_to_tensor(iids,dtype=tf.int64)

In [10]:
iidten

<tf.Tensor: shape=(2455, 512), dtype=int64, numpy=
array([[  101, 26568,  8873, ...,     0,     0,     0],
       [  101,  2273,  2075, ...,     0,     0,     0],
       [  101,  1045,  2969, ...,     0,     0,     0],
       ...,
       [  101,  1042, 10139, ...,     0,     0,     0],
       [  101,  4435, 10993, ...,     0,     0,     0],
       [  101,  1045,  2179, ...,     0,     0,     0]], dtype=int64)>

In [11]:
amsk=mh['attention_mask'].tolist()
amskten=tf.convert_to_tensor(amsk,dtype=tf.int64)

In [12]:
amskten

<tf.Tensor: shape=(2455, 512), dtype=int64, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], dtype=int64)>

In [13]:
ttid=mh['token_type_ids'].tolist()
ttidten=tf.convert_to_tensor(ttid,dtype=tf.int64)

In [14]:
ttidten

<tf.Tensor: shape=(2455, 512), dtype=int64, numpy=
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)>

In [15]:
lbl=mh['status_encodes'].tolist()
lblten=tf.convert_to_tensor(lbl)

In [16]:
lblten

<tf.Tensor: shape=(2455,), dtype=float32, numpy=array([0., 0., 2., ..., 0., 3., 2.], dtype=float32)>

In [17]:
batch_size=16
ttiset=tf.data.Dataset.from_tensor_slices(ttidten)
ttiset=ttiset.batch(batch_size, drop_remainder=False)

In [18]:
ttiset

<_BatchDataset element_spec=TensorSpec(shape=(None, 512), dtype=tf.int64, name=None)>

In [19]:
idset=tf.data.Dataset.from_tensor_slices(iidten)
idset=idset.batch(batch_size, drop_remainder=False)

In [20]:
idset

<_BatchDataset element_spec=TensorSpec(shape=(None, 512), dtype=tf.int64, name=None)>

In [21]:
amset=tf.data.Dataset.from_tensor_slices(amskten)
amset=amset.batch(batch_size, drop_remainder=False)

In [22]:
amset

<_BatchDataset element_spec=TensorSpec(shape=(None, 512), dtype=tf.int64, name=None)>

In [23]:
lblset=tf.data.Dataset.from_tensor_slices(lblten)
lblset=lblset.batch(batch_size, drop_remainder=False)

In [24]:
lblset

<_BatchDataset element_spec=TensorSpec(shape=(None,), dtype=tf.float32, name=None)>

In [25]:
combined_dataset=tf.data.Dataset.zip((idset, amset, ttiset, lblset))

In [26]:
# Shuffle the dataset and take 100 elements
combined_dataset1 = combined_dataset.shuffle(buffer_size=1000).take(100)

In [27]:
for batch in combined_dataset1:
    print([b.shape for b in batch])

[TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16])]
[TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16])]
[TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16])]
[TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16])]
[TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16])]
[TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16])]
[TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16])]
[TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16])]
[TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16])]
[TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16])]
[TensorShape([16, 512]), TensorShape([16, 512]), TensorShape([16, 512]), TensorS

In [28]:
combined_dataset1

<_TakeDataset element_spec=(TensorSpec(shape=(None, 512), dtype=tf.int64, name=None), TensorSpec(shape=(None, 512), dtype=tf.int64, name=None), TensorSpec(shape=(None, 512), dtype=tf.int64, name=None), TensorSpec(shape=(None,), dtype=tf.float32, name=None))>

In [29]:
def convert_to_bert_format(batch):
    input_ids, attention_mask, token_type_ids, labels = batch
    
    # Prepare the dictionary of BERT inputs
    bert_inputs = {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'token_type_ids': token_type_ids
    }
    
    # Return BERT input dictionary along with the labels
    return bert_inputs, labels

# Apply the mapping function to the dataset
bert_dataset = combined_dataset1.map(lambda *batch: convert_to_bert_format(batch))

In [30]:
bert_dataset

<_MapDataset element_spec=({'input_ids': TensorSpec(shape=(None, 512), dtype=tf.int64, name=None), 'attention_mask': TensorSpec(shape=(None, 512), dtype=tf.int64, name=None), 'token_type_ids': TensorSpec(shape=(None, 512), dtype=tf.int64, name=None)}, TensorSpec(shape=(None,), dtype=tf.float32, name=None))>

In [31]:
for batch in bert_dataset.take(1):
    print(batch)

({'input_ids': <tf.Tensor: shape=(16, 512), dtype=int64, numpy=
array([[ 101, 2026, 3105, ...,    0,    0,    0],
       [ 101, 1044, 6672, ...,    0,    0,    0],
       [ 101, 1045, 2074, ...,    0,    0,    0],
       ...,
       [ 101, 1045, 2572, ...,    0,    0,    0],
       [ 101, 1045, 4149, ...,    0,    0,    0],
       [ 101, 1045, 2079, ...,    0,    0,    0]], dtype=int64)>, 'attention_mask': <tf.Tensor: shape=(16, 512), dtype=int64, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], dtype=int64)>, 'token_type_ids': <tf.Tensor: shape=(16, 512), dtype=int64, numpy=
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)>}, <tf.Tensor: shape=(16,), dtype=fl

In [34]:
import tensorflow as tf
from transformers import TFAutoModel
import warnings
warnings.filterwarnings("ignore")

In [35]:
model = TFAutoModel.from_pretrained("bert-base-uncased")

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [36]:
class BERTForClassification(tf.keras.Model):
    
    def __init__(self, bert_model, num_classes):
        super().__init__()
        self.bert = bert_model
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax')
        
    def call(self, inputs):
        x = self.bert(inputs)[1]
        return self.fc(x)

In [37]:
classifier = BERTForClassification(model, num_classes=7)

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [38]:
history = classifier.fit(
    bert_dataset,
    epochs=3
)

Epoch 1/3


100/100 [==============================] - 7671s 77s/step - loss: 1.3164 - accuracy: 0.5242
Epoch 2/3
100/100 [==============================] - 7355s 74s/step - loss: 0.9790 - accuracy: 0.5852
Epoch 3/3
100/100 [==============================] - 7288s 73s/step - loss: 0.8464 - accuracy: 0.6518
